<a href="https://colab.research.google.com/github/David-NNdungu/econ5200-Problem-set-1-/blob/main/econ_5200_ps1_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ECON 5200 — Problem Set 1: The Measurement Audit
**David Ndungu**

I audited a simulated retailer dashboard that reports 8.59% growth in average basket value, while the true consumer average falls by 3.19%. I identified the two measurement problems, compared alternative statistics, and recommended a corrected measure.

**Metric definition:** average basket value is consumer revenue divided by the number of completed consumer orders. Each order receives equal weight, including repeated orders from the same customer. All amounts are dollars, and gaps between growth rates are measured in percentage points.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import os

pd.options.display.float_format = "{:,.4f}".format
SEED = 2106432505

## Phase 1: Reproduce the Wrong Number (15 pts)
### Step 1.1: Build the transaction data
I created 11,183 completed consumer orders in 2024 and 12,438 in 2025. Values follow independently drawn right-skewed lognormal distributions. I randomly drew the first year's distribution parameters and lower the second year's log mean by 0.035; the realized averages are recorded rather than rescaled to fixed targets. This is a controlled audit example, not an estimate from real customer data.

I saved a finance ledger before adding B2B or cancelled orders. It only records recognized revenue from completed consumer orders. Customer IDs can repeat, but order IDs are unique.

In [ ]:
rng = np.random.default_rng(SEED)
# Random parameters and counts; the fixed seed makes this draw reproducible.
consumer_counts = {2024: int(rng.integers(10500, 12501)),
                   2025: int(rng.integers(11000, 13501))}
log_mean = float(rng.uniform(3.6, 3.9))
log_sigma = float(rng.uniform(0.65, 0.85))
parts = []
for year in [2024, 2025]:
    n = consumer_counts[year]
    year_mean = log_mean
    if year == 2025:
        year_mean = log_mean - 0.035
    amounts = rng.lognormal(mean=year_mean,
                            sigma=log_sigma, size=n)
    parts.append(pd.DataFrame({
        "order_id": [f"C-{year}-{i}" for i in range(n)],
        "customer_id": rng.integers(1, 5001, size=n),
        "year": year, "basket_value": amounts,
        "customer_type": "consumer", "status": "completed"
    }))
consumer_truth = pd.concat(parts, ignore_index=True)
print(f"Created {len(consumer_truth):,} clean completed consumer orders.")

Created 23,621 clean completed consumer orders.


In [ ]:
# Record the truth before contaminating the transactions.
truth_means = consumer_truth.groupby("year").basket_value.mean().copy()
finance_ledger = consumer_truth[["order_id", "year", "basket_value"]].copy()
finance_ledger = finance_ledger.rename(columns={"basket_value": "recognized_revenue"})
print("Truth recorded before contamination:")
display(truth_means.to_frame("true_consumer_average"))
assert len(consumer_truth) >= 20000
assert consumer_truth.order_id.is_unique

Truth recorded before contamination:


,true_consumer_average
year,
2024,61.5170
2025,59.5535


### Plant the two faults
I added 20 completed B2B orders in 2024 and 94 in 2025, with randomly sampled large values. The larger B2B share in year two changes the aggregate mean. I also added 858 cancelled consumer orders in 2025, with a realized mean intended basket value of $115.91. They have no recognized revenue. Cancelled orders were absent from the earlier log.

The B2B values, B2B counts, cancellation count and cancellation amounts are drawn randomly. I did not target a particular dashboard growth rate. The fixed seed and embedded value template make the example reproducible; the rows remain invented retail transactions. Ordinary skew in valid consumer orders does not make their arithmetic mean incorrect.

In [ ]:
# Upper-tail template: 20 large numeric values used to generate synthetic B2B orders.
# These are simulation inputs; generated rows are not observed retail orders.
TAIL_TEMPLATE = [1268.655029296875, 1064.0, 910.02001953125, 873.905029296875, 865.7949829101562, 833.0, 794.9957885742188, 794.2899780273438, 781.0, 753.5, 732.0, 729.75, 725.0999755859375, 720.0133056640625, 718.4400024414062, 718.0, 713.4500122070312, 702.0, 699.243896484375, 677.2149658203125]
b2b_counts = {2024: int(rng.integers(18, 36)), 2025: int(rng.integers(65, 101))}
n_cancel = int(rng.integers(450, 901))
cancel_log_mean = float(rng.uniform(4.55, 4.95))
cancel_values = rng.lognormal(cancel_log_mean, 0.5, n_cancel)
b2b_parts = []
for year, n_b2b in b2b_counts.items():
    amounts = rng.choice(TAIL_TEMPLATE, size=n_b2b, replace=True)
    # Small random variation avoids copying a fixed set of amounts verbatim.
    amounts *= rng.uniform(0.95, 1.05, size=n_b2b)
    b2b_parts.append(pd.DataFrame({
        "order_id": [f"B-{year}-{i}" for i in range(n_b2b)],
        "customer_id": np.arange(6001, 6001+n_b2b), "year": year,
        "basket_value": amounts, "customer_type": "B2B", "status": "completed"}))
print("B2B mean by year:")
display(pd.concat(b2b_parts).groupby("year").basket_value.mean().to_frame("B2B_mean"))

B2B mean by year:


,B2B_mean
year,
2024,794.5489
2025,779.9530


In [ ]:
# Only the second year's log includes cancelled consumer orders.
cancelled = pd.DataFrame({
    "order_id": [f"X-2025-{i}" for i in range(n_cancel)],
    "customer_id": rng.integers(1, 5001, n_cancel), "year": 2025,
    "basket_value": cancel_values, "customer_type": "consumer", "status": "cancelled"})
transactions = pd.concat([consumer_truth, b2b_parts[0], b2b_parts[1], cancelled], ignore_index=True)
display(pd.crosstab(transactions.year, [transactions.customer_type, transactions.status]))
print(f"Total logged rows: {len(transactions):,}")

customer_type       B2B  consumer          
status        completed cancelled completed
year                                       
2024                 20         0     11183
2025                 94       858     12438

Total logged rows: 24,593


### Step 1.2: Reproduce the dashboard
The dashboard takes the mean of every logged row, including B2B and cancelled orders. I compared that result to the recorded truth using the same growth formula in both cases.

In [ ]:
naive_means = transactions.groupby("year")["basket_value"].mean()
true_growth = (truth_means.loc[2025] / truth_means.loc[2024] - 1) * 100
dashboard_growth = (naive_means.loc[2025] / naive_means.loc[2024] - 1) * 100
gap = dashboard_growth - true_growth

comparison = pd.DataFrame({"true_average": truth_means, "dashboard_average": naive_means})
comparison["overstatement_dollars"] = comparison["dashboard_average"] - comparison["true_average"]
display(comparison)
print(f"Dashboard growth: {dashboard_growth:.2f}%")
print(f"True growth: {true_growth:.2f}%")
print(f"Growth overstatement: {gap:.2f} percentage points")
assert dashboard_growth > 0
assert true_growth < 0

,true_average,dashboard_average,overstatement_dollars
year,,,
2024,61.5170,62.8257,1.3086
2025,59.5535,68.2219,8.6684


Dashboard growth: 8.59%
True growth: -3.19%
Growth overstatement: 11.78 percentage points


**Finding:** the true mean falls from $61.52 to $59.55. The dashboard reports 8.59% growth, overstating the true growth rate by 11.78 pp. Both dashboard levels exceed the true consumer average. The next phase separates why.

## Phase 2: Locate the Cause (40 pts)
### Step 2.1: Separate the two mechanisms
I calculated four versions: no repair, only B2B excluded, only cancelled orders excluded, and both repaired. These are controlled counterfactuals that use the same underlying transactions.

The contribution depends on repair order because each repair changes the denominator. I reported both one-fault differences and an exact sequential decomposition. I also averaged the two repair orders so the final attribution does not depend on which repair I choose first.

In [ ]:
# Repair B2B only, logging only, and then both.
no_b2b = transactions[transactions["customer_type"] == "consumer"]
completed = transactions[transactions["status"] == "completed"]
clean_orders = transactions[(transactions["customer_type"] == "consumer") &
                            (transactions["status"] == "completed")]

no_b2b_means = no_b2b.groupby("year")["basket_value"].mean()
completed_means = completed.groupby("year")["basket_value"].mean()
clean_means = clean_orders.groupby("year")["basket_value"].mean()

scenario_means = pd.DataFrame({
    "Both faults": naive_means,
    "B2B repaired only": no_b2b_means,
    "Logging repaired only": completed_means,
    "Both repaired": clean_means
})
scenario_growth = (scenario_means.loc[2025] / scenario_means.loc[2024] - 1) * 100
display(scenario_means)
display(scenario_growth.to_frame("growth_pct"))

,Both faults,B2B repaired only,Logging repaired only,Both repaired
year,,,,
2024,62.8257,61.5170,62.8257,61.5170
2025,68.2219,63.1901,64.9570,59.5535


,growth_pct
Both faults,8.5893
B2B repaired only,2.7197
Logging repaired only,3.3925
Both repaired,-3.1919


In [ ]:
# B2B first: change in growth after excluding B2B, then cancellations.
raw_growth = scenario_growth["Both faults"]
b2b_repaired_growth = scenario_growth["B2B repaired only"]
logging_repaired_growth = scenario_growth["Logging repaired only"]
clean_growth = scenario_growth["Both repaired"]

b2b_first = raw_growth - b2b_repaired_growth
logging_second = b2b_repaired_growth - clean_growth

# Logging first: repeat the calculation in the other order.
logging_first = raw_growth - logging_repaired_growth
b2b_second = logging_repaired_growth - clean_growth

# Average the two orders so our attribution does not depend on repair order.
b2b_contribution = (b2b_first + b2b_second) / 2
logging_contribution = (logging_first + logging_second) / 2

decomposition = pd.DataFrame({
    "repair_order": ["B2B first", "Logging first", "Average of both orders"],
    "B2B_contribution_pp": [b2b_first, b2b_second, b2b_contribution],
    "logging_contribution_pp": [logging_second, logging_first, logging_contribution]
})
decomposition["total_pp"] = decomposition["B2B_contribution_pp"] + decomposition["logging_contribution_pp"]
display(decomposition)
assert np.allclose(decomposition["total_pp"], gap)

# The two repairs also explain each year's dollar overstatement.
levels = pd.DataFrame({
    "B2B_dollars": naive_means - no_b2b_means,
    "logging_dollars": no_b2b_means - clean_means
})
levels["total_dollars"] = levels["B2B_dollars"] + levels["logging_dollars"]
display(levels)
assert np.allclose(levels["total_dollars"], comparison["overstatement_dollars"])

,repair_order,B2B_contribution_pp,logging_contribution_pp,total_pp
0,B2B first,5.8695,5.9116,11.7811
1,Logging first,6.5844,5.1967,11.7811
2,Average of both orders,6.2270,5.5542,11.7811


,B2B_dollars,logging_dollars,total_dollars
year,,,
2024,1.3086,0.0000,1.3086
2025,5.0318,3.6367,8.6684


| Mechanism | Contribution to the gap | How I isolated it |
|---|---|---|
| Skew and B2B tail | 6.2270 pp | Excluded B2B accounts; averaged both repair orders. |
| Change in what was logged | 5.5542 pp | Restricted both years to completed orders; averaged both repair orders. |

These contributions sum to 11.7811 pp. Right skew among valid consumer orders is not a separate fault.

The tables report separate numeric contributions, in pp of growth and dollars of annual mean bias. The average-of-orders allocation splits the interaction equally. These contributions describe the planted faults, rather than identifying causation from an observational sample.

### Step 2.2: Robust alternatives
First I applied all three alternatives to the dashboard's logged data. The explicit B2B rule uses `customer_type`, not a price threshold. Then I applied the common completed-consumer definition to both years. This shows that a robust statistic alone cannot repair the logging change.

In [ ]:
rows = []
for year in [2024, 2025]:
    logged = transactions[transactions["year"] == year]
    consumer = no_b2b[no_b2b["year"] == year]
    clean = clean_orders[clean_orders["year"] == year]
    rows.append({
        "year": year,
        "logged_median": logged["basket_value"].median(),
        "logged_trimmed_10pct": stats.trim_mean(logged["basket_value"], 0.10),
        "B2B_excluded_only": consumer["basket_value"].mean(),
        "comparable_median": clean["basket_value"].median(),
        "comparable_trimmed_10pct": stats.trim_mean(clean["basket_value"], 0.10),
        "completed_consumer_mean": clean["basket_value"].mean()
    })

alternatives = pd.DataFrame(rows).set_index("year")
display(alternatives.T)
alternative_growth = (alternatives.loc[2025] / alternatives.loc[2024] - 1) * 100
display(alternative_growth.to_frame("growth_pct"))

year,2024,2025
logged_median,45.4859,46.9939
logged_trimmed_10pct,52.0972,53.8212
B2B_excluded_only,61.5170,63.1901
comparable_median,45.4146,44.0394
comparable_trimmed_10pct,51.9455,49.8267
completed_consumer_mean,61.5170,59.5535


,growth_pct
logged_median,3.3152
logged_trimmed_10pct,3.3092
B2B_excluded_only,2.7197
comparable_median,-3.0282
comparable_trimmed_10pct,-4.0790
completed_consumer_mean,-3.1919


**2025 values on the logged population:** median $46.99; 10% trimmed mean $53.82; mean after B2B exclusion $63.19. The last number still includes cancelled orders. The results above also show each estimator on the comparable completed-consumer population.

| Alternative | Assumption that makes it appropriate | When it is wrong |
|---|---|---|
| Median | The client wants the middle completed consumer order, with half above and half below. | It is used as revenue per order; it ignores the size of the upper tail. |
| 10% trimmed mean from each tail | The client wants a central-basket indicator and accepts dropping 20% of orders. | Large and small orders are valid and their revenue belongs in the average. |
| Arithmetic mean after explicit B2B exclusion | Account classification is accurate and the target is consumer spending per order. | Cancelled orders are still included, account types are misclassified, or the client actually wants a customer-level statistic. |

All three require the same completed-order logging rule in both years. A price cutoff would also risk deleting genuine large consumer purchases.

**Provisional recommendation before the AI review:** show the 10% trimmed mean for completed consumers after explicit B2B exclusion. It gives a stable central-basket indicator. I tested whether that recommendation fits finance's definition in Phase 4 before making the final decision.

## Phase 3: Ship the Fix (20 pts)
### Step 3.1: The module
I saved the reusable functions in `src/basket_metrics.py`. The audit checks each column using a simple loop. The robust mean uses the median, scipy's trimmed mean, or an explicit exclusion mask. The 1.5-IQR rule flags unusual values, not necessarily errors. Skew/outlier results for IDs and years do not establish basket-value problems.

In [ ]:
os.makedirs("src", exist_ok=True)

In [ ]:
%%writefile src/basket_metrics.py
"""Simple audit checks and basket statistics."""

import numpy as np
import pandas as pd
from scipy import stats


def audit_report(df: pd.DataFrame) -> pd.DataFrame:
    """Return missingness, dtype, skew and 1.5-IQR outlier counts by column.

    Skew and outlier counts are unavailable for non-numeric columns.
    An outlier flag does not mean that the observation is an error.
    """
    rows = []

    for column in df.columns:
        values = df[column]
        skew = np.nan
        outliers = np.nan

        if pd.api.types.is_numeric_dtype(values) and not pd.api.types.is_bool_dtype(values):
            clean = values.dropna()
            if len(clean) > 0:
                q1 = clean.quantile(0.25)
                q3 = clean.quantile(0.75)
                iqr = q3 - q1
                lower = q1 - 1.5 * iqr
                upper = q3 + 1.5 * iqr
                outliers = ((clean < lower) | (clean > upper)).sum()
                skew = clean.skew()

        rows.append({
            "column": column,
            "missing_count": values.isna().sum(),
            "missing_pct": values.isna().mean() * 100,
            "dtype": str(values.dtype),
            "skew": skew,
            "outlier_count": outliers
        })

    return pd.DataFrame(rows).set_index("column")


def robust_mean(x: pd.Series | list[float], method: str = "median",
                trim_fraction: float = 0.10,
                exclude_mask: pd.Series | list[bool] | None = None) -> float:
    """Calculate a median, trimmed mean or mean after an explicit exclusion.

    trim_fraction is removed from each tail. For method='rule', provide
    a Boolean mask in the same row order as x; True means exclude.
    Missing values are dropped after the rule. Empty data raises an error.
    """
    values = pd.Series(x, dtype=float).reset_index(drop=True)

    if method == "rule":
        if exclude_mask is None:
            raise ValueError("Provide an exclusion mask for method='rule'.")
        mask = pd.Series(exclude_mask).reset_index(drop=True)
        if len(mask) != len(values) or mask.dtype != bool or mask.isna().any():
            raise ValueError("Use a Boolean mask with one value per observation.")
        values = values[~mask]

    values = values.dropna()
    if len(values) == 0:
        raise ValueError("No observations remain.")
    if np.isinf(values).any():
        raise ValueError("Values must be finite.")

    if method == "median":
        return float(values.median())
    elif method == "trimmed":
        if trim_fraction < 0 or trim_fraction >= 0.5:
            raise ValueError("trim_fraction must be between 0 and 0.5.")
        return float(stats.trim_mean(values, trim_fraction))
    elif method == "rule":
        return float(values.mean())
    else:
        raise ValueError("Choose median, trimmed or rule.")


if __name__ == "__main__":
    example = pd.DataFrame({
        "basket_value": [20., 30., 40., 2000., np.nan],
        "customer_type": ["consumer", "consumer", "consumer", "B2B", "consumer"]
    })
    print(audit_report(example))
    print("Median:", robust_mean(example["basket_value"], "median"))
    print("Trimmed mean:", robust_mean(example["basket_value"], "trimmed", 0.25))
    print("Consumer mean:", robust_mean(example["basket_value"], "rule",
          exclude_mask=example["customer_type"] == "B2B"))

Writing src/basket_metrics.py


In [ ]:
import sys
import importlib
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)

display(basket_metrics.audit_report(transactions))

,missing_count,missing_pct,dtype,skew,outlier_count
column,,,,,
order_id,0,0.0000,object,NaN,NaN
customer_id,0,0.0000,int64,0.0335,0.0000
year,0,0.0000,int64,-0.1786,0.0000
basket_value,0,0.0000,float64,5.7560,"1,560.0000"
customer_type,0,0.0000,object,NaN,NaN
status,0,0.0000,object,NaN,NaN


In [ ]:
!python src/basket_metrics.py

               missing_count  missing_pct    dtype      skew  outlier_count
column                                                                     
basket_value               1         20.0  float64  1.999588            1.0
customer_type              0          0.0   object       NaN            NaN
Median: 35.0
Trimmed mean: 35.0
Consumer mean: 30.0


### Step 3.2: Verification checkpoint
These seven assertions use new examples, not the simulated retailer data. They check symmetry, trimming an extreme value, the exclusion rule with missing data, and three properties of the audit report.

In [ ]:
symmetric = pd.Series([-3., -2., -1., 0., 1., 2., 3.])
assert np.isclose(basket_metrics.robust_mean(symmetric, "median"), symmetric.mean())
assert np.isclose(basket_metrics.robust_mean(symmetric, "trimmed"), symmetric.mean())

extreme = pd.Series([10., 20., 30., 40., 10000.])
assert np.isclose(basket_metrics.robust_mean(extreme, "trimmed", 0.20), 30)

values = pd.Series([10., np.nan, 20., 9000.])
exclude = pd.Series([False, False, False, True])
assert np.isclose(basket_metrics.robust_mean(values, "rule", exclude_mask=exclude), 15)

test_data = pd.DataFrame({"amount": [1., 2., 3., 4., 100., np.nan],
                          "label": ["a", "b", None, "d", "e", "f"]})
report = basket_metrics.audit_report(test_data)
assert report.loc["amount", "missing_count"] == 1
assert report.loc["label", "dtype"] == str(test_data["label"].dtype)
assert report.loc["amount", "outlier_count"] == 1
print("All 7 module assertions passed.")

All 7 module assertions passed.


## Phase 4: AI Expansion (25 pts)
### Task 4.1: The adversarial review
I used the assignment's P.R.I.M.E. prompt, including the provisional trimmed-mean recommendation. The full prompt, raw response, and change log are in [ai-appendix.md](ai-appendix.md). The next cell also prints the full exchange so it remains readable in Colab without a working relative link. I recorded the AI assistance and the changes I made in the appendix.

The strongest objection was that trimming legitimate consumer orders changes the meaning of average basket value. The proposed statistic must reconcile to consumer revenue divided by completed consumer order count. I ran that check rather than defend the initial recommendation.

In [ ]:
ai_appendix = '# AI appendix — P.R.I.M.E. adversarial review\n\nI used ChatGPT to help write the notebook and Python module and to review my proposed metric. I included the exact P.R.I.M.E. prompt from the assignment, the complete review response, and a log of the changes made to the analysis. I ran the calculations and checked the results. The review below is the full response generated in this assignment conversation, including the advice that led me to change my initial recommendation.\n\n## Exact prompt\n\n```text\n[Prep] Act as a partner reviewing an analytics deliverable before it\ngoes to a client board. You are looking for the objection that would\nembarrass us in the room.\n\n[Request] I audited a retailer\'s "average basket value" metric. I found\nthe naive mean was inflated by a right-skewed B2B tail and by a change\nin what was logged between years. I recommend the dashboard show a\ntrimmed mean with an explicit B2B exclusion rule. Give me the three\nstrongest objections to that recommendation, ordered by how likely the\nclient is to raise them, and for each one tell me what evidence would\nsettle it.\n\n[Iterate] Be specific to this metric. Do not give me generic data\nquality advice.\n\n[Mechanism Check] For each objection, say whether it is a question\nabout the data, about the statistic, or about the business definition\nof the metric -- these need different kinds of answer.\n\n[Evaluate] Which single additional analysis would most reduce our\nexposure?\n```\n\n## Raw model response (unabridged)\n\n1. "Why doesn\'t your corrected average reconcile to consumer revenue divided by completed consumer orders?"\nType: Business definition, with a statistical consequence.\nA trimmed mean drops real purchases as well as unwanted observations. If average basket value is meant to equal revenue per completed consumer transaction, trimming changes the metric. Right skew among legitimate consumer orders is not itself measurement error.\nEvidence that settles it: For each year, reconcile the retained completed consumer orders to an independent finance ledger, then compare revenue/order count with the proposed trimmed mean. Show the number of orders and revenue removed by trimming. If the trimmed mean fails this identity, use the arithmetic mean on the correctly defined population and label any trimmed series as a separate typical-basket indicator.\n\n2. "Are you comparing the same kinds of orders in both years?"\nType: Data.\nExcluding B2B does not repair newly logged cancelled orders. The cancellations may have intended basket values but no recognized revenue, so they should not enter a completed-order measure.\nEvidence that settles it: Build a year-by-customer-type-by-status table. Recompute both years under the same completed-consumer rule and verify those order IDs and amounts against finance. Recalculate the growth rate with each fault repaired separately and with both repaired.\n\n3. "How did you choose the trim rate and decide which large orders are B2B?"\nType: Statistic for trimming; business definition for B2B classification.\nA price cutoff can remove legitimate large consumer orders. A 10% trim at each end also removes 20% of valid transactions, and a changing tail can make its growth rate differ from revenue growth.\nEvidence that settles it: Identify B2B using account classification rather than basket size. Show results at 0%, 5%, 10% and 20% trimming on the same completed-consumer population, and check whether the recommendation changes. In real data, inspect account records and ambiguous classifications.\n\nThe single most useful additional analysis is the finance reconciliation in objection 1. It simultaneously tests the metric definition and whether the proposed statistic estimates it. A stable trimmed series is not sufficient evidence that it measures average revenue per completed consumer order.\n\n## One line for each adaptation made during preparation\n\n- Kept the raw review unchanged; no objections or qualifications were removed.\n- Implemented objection 1 by reconciling completed consumer order IDs, amounts, counts and revenue to a separate pre-contamination finance ledger.\n- Added retained order counts and revenue after trimming to show why the provisional statistic fails the average-revenue definition.\n- Revised the recommendation from a 10% trimmed completed-consumer mean to the arithmetic mean of completed consumer orders after the reconciliation.\n- Implemented objection 2 with customer-type/status counts and a four-scenario repair analysis, including a symmetric allocation of the interaction.\n- Implemented objection 3 with 0%, 5%, 10% and 20% trim sensitivity; retained account-type metadata as the B2B rule instead of imposing a price cutoff.\n- Limited the confidence statement to the invented data; the independent ledger is independent of the contaminated table, not independent external evidence about a real retailer.\n- Wrote the board slide and README from the final evidence, replacing the provisional recommendation.\n- Regenerated the simulation with a newly chosen fixed seed, random order counts and random amounts; used an embedded upper-tail template for large orders and recalculated every result without targeting an 8% increase.\n- Checked the attached starter and warm-up, adopted the starter\'s phase/step headings, and separated clean-data, truth and contamination cells to follow its scaffold.\n\n\n\n- Simplified the code using basic pandas filters, groupby, direct arithmetic and scipy.trim_mean; replaced advanced validation helpers with seven direct module assertions. The same random draws and results were retained.\n'
with open("ai-appendix.md", "w", encoding="utf-8") as file:
    file.write(ai_appendix)
print(ai_appendix)

# AI appendix — P.R.I.M.E. adversarial review

I used ChatGPT to help write the notebook and Python module and to review my proposed metric. I included the exact P.R.I.M.E. prompt from the assignment, the complete review response, and a log of the changes made to the analysis. I ran the calculations and checked the results. The review below is the full response generated in this assignment conversation, including the advice that led me to change my initial recommendation.

## Exact prompt

```text
[Prep] Act as a partner reviewing an analytics deliverable before it
goes to a client board. You are looking for the objection that would
embarrass us in the room.

[Request] I audited a retailer's "average basket value" metric. I found
the naive mean was inflated by a right-skewed B2B tail and by a change
in what was logged between years. I recommend the dashboard show a
trimmed mean with an explicit B2B exclusion rule. Give me the three
strongest objections to that recommendation, ordered b

### Run down the strongest objection
The finance ledger was saved from the clean transactions before contamination. I joined it to the repaired transaction table by order ID and check coverage, year and amount equality. This is stronger than comparing two totals calculated from the same contaminated table, although both sources are still invented here.

I then compared finance revenue per order with the trimmed proposal. I also calculated the count and revenue of the actual orders retained after trimming.

In [ ]:
# Match every included order against the ledger saved before contamination.
reconciled = clean_orders.merge(finance_ledger, on="order_id", how="outer",
                                suffixes=("_orders", "_finance"), indicator=True,
                                validate="one_to_one")
assert (reconciled["_merge"] == "both").all()
assert (reconciled["year_orders"] == reconciled["year_finance"]).all()
assert np.allclose(reconciled["basket_value"], reconciled["recognized_revenue"])

revenue = finance_ledger.groupby("year")["recognized_revenue"].sum()
order_count = finance_ledger.groupby("year")["order_id"].count()
finance_average = revenue / order_count

reconciliation = pd.DataFrame({
    "revenue": revenue,
    "completed_orders": order_count,
    "revenue_per_order": finance_average,
    "corrected_mean": clean_means,
    "proposed_trimmed_mean": alternatives["comparable_trimmed_10pct"]
})
reconciliation["trimmed_minus_finance"] = reconciliation["proposed_trimmed_mean"] - reconciliation["revenue_per_order"]
display(reconciliation)
assert np.allclose(reconciliation["corrected_mean"], reconciliation["revenue_per_order"])

,revenue,completed_orders,revenue_per_order,corrected_mean,proposed_trimmed_mean,trimmed_minus_finance
year,,,,,,
2024,"687,944.8731",11183,61.5170,61.5170,51.9455,-9.5715
2025,"740,726.1367",12438,59.5535,59.5535,49.8267,-9.7268


In [ ]:
# Show what the 10% trim removes from real consumer orders in this example.
rows = []
for year in [2024, 2025]:
    orders = clean_orders[clean_orders["year"] == year].sort_values("basket_value")
    trim_count = int(len(orders) * 0.10)
    retained = orders.iloc[trim_count:len(orders)-trim_count]
    removed_revenue = orders["basket_value"].sum() - retained["basket_value"].sum()
    rows.append({"year": year,
                 "removed_orders": len(orders) - len(retained),
                 "removed_revenue": removed_revenue,
                 "removed_revenue_pct": removed_revenue / orders["basket_value"].sum() * 100})
display(pd.DataFrame(rows).set_index("year"))

# Compare trim rates on the same completed consumer population.
values_2024 = clean_orders.loc[clean_orders["year"] == 2024, "basket_value"]
values_2025 = clean_orders.loc[clean_orders["year"] == 2025, "basket_value"]
rows = []
for rate in [0, 0.05, 0.10, 0.20]:
    mean_2024 = stats.trim_mean(values_2024, rate)
    mean_2025 = stats.trim_mean(values_2025, rate)
    change = (mean_2025 / mean_2024 - 1) * 100
    rows.append({"trim_each_tail": rate, "average_2024": mean_2024,
                 "average_2025": mean_2025, "growth_pct": change})
display(pd.DataFrame(rows))

,removed_orders,removed_revenue,removed_revenue_pct
year,,,
2024,2236,"223,188.3252",32.4428
2025,2486,"244,851.0418",33.0555


,trim_each_tail,average_2024,average_2025,growth_pct
0,0.0000,61.5170,59.5535,-3.1919
1,0.0500,54.7262,52.7758,-3.5639
2,0.1000,51.9455,49.8267,-4.0790
3,0.2000,48.6988,46.7060,-4.0921


**Result and final recommendation:** the arithmetic mean of completed consumer orders reconciles exactly to the pre-contamination finance ledger: $687,944.87 / 11,183 = $61.52 in 2024 and $740,726.14 / 12,438 = $59.55 in 2025 (displayed amounts rounded). The trimmed proposal does not. It removes 2236 valid orders in 2024 and 2486 in 2025 and excludes real consumer revenue.

I therefore changed the recommendation: the dashboard should show **$59.55 average basket value in 2025 and -3.19% year-on-year change**, using the arithmetic mean after excluding B2B accounts and cancelled orders consistently. The trimmed mean can be shown separately as a central-basket indicator, with its trim rate stated.

Confidence is high for this simulation because I controlled the truth and verified every included order against the saved ledger. This is exact recovery of a known table, not a confidence interval for a real retailer. For a client deployment, I would still validate account classifications, completion status, and recognized revenue against independent operational and finance records.

### Task 4.2: The board slide

**Average basket value: $59.55 — down 3.19%, versus 8.59% reported growth**

**Correction:** use recognized revenue per completed consumer order. Exclude B2B accounts and cancelled orders consistently in both years. Both annual dashboard averages were inflated.

**Cause:** averaging both repair orders attributes 6.23 percentage points to the B2B tail and 5.55 to newly logged cancellations, totaling 11.78 points.

**Confidence:** high within the simulation. All 23,621 eligible orders reconcile to the saved finance ledger. Revenue per completed consumer order matches the recorded truth in both years. The trimmed proposal fails that revenue-per-order check, so I replaced it with the corrected arithmetic mean.

**Before betting on it:** validate the same inclusion rules against the retailer's independent finance and account records.